<a id='toc0_'></a>
## Content Outline

### Intro
- [The Backbone Matrix](#toc1_)
- [Short standalone versions](#toc1_1_1_1_)
- [A display utility for the backbone matrix](#toc1_1_2_)

### Sequences
- [Riordan](#toc1_2_)
- [Catalan](#toc1_3_)
- [Motzkin](#toc1_4_)
- [Fibonacci](#toc1_5_)
- [Factorial](#toc1_6_)
- [Kolakoski](#toc1_7_)
- [Bell](#toc1_8_)
- [Fubini](#toc1_9_)
- [Central Binomial](#toc1_10_)
- [Subfactorial / Rencontres / Derangements](#toc1_11_)
- [Involutions (Young tableaux with n cells)](#toc1_12_)
- [Moebius](#toc1_13_)
- [Pell](#toc1_14_)
- [Jacobsthal](#toc1_15_)
- [Polya Trees](#toc1_16_)
- [Euler (alternating permutations, boustrophedon transform)](#toc1_17_)
- [Little Schroeder numbers](#toc1_18_)
- [Big Schroeder numbers](#toc1_19_)
- [Central Delannoy](#toc1_20_)
- [Sets of Lists](#toc1_21_)
- [Total Partitions / Ward Set](#toc1_22_)
- [Ward Cycle](#toc1_23_)

# <a id='toc1_'></a>The Backbone Matrix

In [ ]:
from typing import Any, Generator
from itertools import islice

type Seq = list[int]
type Matrix = list[Seq]
type SeqGenerator = Generator[int, Any, None]
type MatGenerator = Generator[Seq, Any, None]

def print_list(gen: SeqGenerator, n: int) -> None:
    seq: Seq = list(islice(gen, n))
    print(seq)
    
def print_matrix(gen: MatGenerator, n: int) -> None:
    mat: Matrix = list(islice(gen, n))
    print(mat)

In [ ]:
class BackboneMatrix:
    def __init__(self, seq: SeqGenerator, dim: int = 0) -> None:
        """
        Matrix builder with a single source matrix.

        The full matrix stores the triangles:
        - lower triangle values in row slices matrix[i][:i+1]
        - upper triangle values in column slices matrix[:j+1][j]
        - antidiagonal values in matrix[i][d-i] for d = 0..n-1
        Since also the antidiagonal-ups, and antidiagonal-downs can be called,
        this class provides a single source for five representations.
        """
        self.seq_source = seq
        self.matrix: Matrix = []

        if dim > 0: self.grow(dim)

    def grow(self, steps: int = 1) -> None:
        """Expands the matrix dimensions by a given number of steps."""
        for _ in range(steps):
            try:
                r = next(self.seq_source)
            except StopIteration:
                break

            n = len(self.matrix)

            if n == 0:
                self.matrix.append([r])
                continue

            # Build the new lower row from the previous lower row slice and r.
            lower_new = self.matrix[-1][:] + [r]
            for k in range(n, 0, -1):
                lower_new[k - 1] += lower_new[k]

            # Build the new upper column from the previous upper column slice and r.
            prev_upper = [self.matrix[i][n - 1] for i in range(n)]
            upper_new = [0] * n + [r]
            for i in range(n - 1, -1, -1):
                upper_new[i] = upper_new[i + 1] - prev_upper[i]

            # Append the new upper values to existing rows.
            for i in range(n):
                self.matrix[i].append(upper_new[i])

            # Append the new lower row.
            self.matrix.append(lower_new)

    @property
    def backbone(self) -> Seq:
        """Returns the backbone of the matrix, which is the main diagonal."""
        n = len(self.matrix)
        return [self.matrix[i][i] for i in range(n)]
    
    @property
    def binomial_invtrans(self) -> Seq:
        """Returns the inverse binomial transform of the backbone, which is the first row."""
        n = len(self.matrix)
        return [self.matrix[0][i] for i in range(n)]
    
    @property
    def binomial_trans(self) -> Seq:
        """Returns the binomial transform of the backbone, which is the first column."""        
        n = len(self.matrix)
        return [self.matrix[i][0] for i in range(n)]
    
    @property
    def lower_rows(self) -> Matrix:
        """Lower triangle as row slices from the single matrix."""
        return [row[: i + 1] for i, row in enumerate(self.matrix)]

    @property
    def lower_rows_sum(self) -> Seq:
        """Row-wise sums of lower_rows."""
        return [sum(row) for row in self.lower_rows]

    @property
    def upper_rows(self) -> Matrix:
        """Upper triangle as column enumerations from the single matrix."""
        n = len(self.matrix)
        return [[self.matrix[i][j] for i in range(j + 1)] for j in range(n)]

    @property
    def upper_rows_sum(self) -> Seq:
        """Row-wise sums of upper_rows."""
        return [sum(row) for row in self.upper_rows]

    @property
    def diagonals_down(self) -> Matrix:
        """
        Returns antidiagonals in descending columns for d = 0..n-1:
        [matrix[0][d], matrix[1][d-1], ..., matrix[d][0]]
        """
        n = len(self.matrix)
        return [[self.matrix[i][d - i] for i in range(d + 1)] for d in range(n)]

    @property
    def diagonals_down_altsum(self) -> Seq:
        """Row-wise alternating sums of diagonals_down."""
        return [sum((-1)**i * row[i] for i in range(len(row))) for row in self.diagonals_down]

    @property
    def diagonals_up(self) -> Matrix:
        """
        Returns antidiagonals in ascending columns for d = 0..n-1:
        [matrix[d][0], matrix[d-1][1], ..., matrix[0][d]]
        """
        n = len(self.matrix)
        return [[self.matrix[d - i][i] for i in range(d + 1)] for d in range(n)]

    @property
    def diagonals_up_sum(self) -> Seq:
        """Row-wise sums of diagonals_up."""
        return [sum(row) for row in self.diagonals_up]

    def get_matrix(self) -> Matrix:
        """Returns the current state of the matrix."""
        return self.matrix

    def __iter__(self):
        return self

    def __next__(self) -> Matrix:
        """
        Allows the class instance to be used directly as a generator.
        Yields a deep copy of the matrix at each expansion step.
        """
        before = len(self.matrix)
        self.grow(1)
        if len(self.matrix) == before:
            raise StopIteration
        return [row[:] for row in self.matrix]

#### <a id='toc1_1_1_1_'></a>Short standalone versions

In [ ]:
def binomial_trans(seq: SeqGenerator, dim: int) -> Seq:
    c = []; t = []
    for _ in range(dim):
        try: r = next(seq)
        except StopIteration: break
        c += [r]
        for i in range(len(c) - 1, 0, -1):
            c[i - 1] += c[i]
        t.append(c[0])
    return t

def invbinomial_trans(seq: SeqGenerator, dim: int) -> Seq:
    c = []; t = []
    for _ in range(dim):
        try: r = next(seq)
        except StopIteration: break
        u = [0] * len(c) + [r]
        for i in range(len(c) - 1, -1, -1): 
            u[i] = u[i + 1] - c[i]
        c = u; t.append(u[0])
    return t

### <a id='toc1_1_2_'></a>A display utility for the backbone matrix.

In [ ]:
def Showcase(seq: SeqGenerator, dim: int = 9, verbose: bool = False) -> None:
    """
    Demonstrates the matrix building process.

    Args:
        seq (SeqGenerator): An iterator providing the sequence of integers.
        dim (int): The initial dimension of the matrix to build.
        verbose (bool): If True, prints also antidiagonal triangles.
    """
    builder = BackboneMatrix(seq, dim)
    
    print("Binomial Matrix")
    for row in builder.get_matrix(): print(row)
    
    print("\nMain Diagonal")
    print([value for value in builder.backbone])
    
    print("\nBinomial Transform")
    print([value for value in builder.binomial_trans])
    
    print("\nInverse Binomial Transform")
    print([value for value in builder.binomial_invtrans])
    
    print("\nLower Triangular")
    for row in builder.lower_rows: print(row)

    print("\nLower Triangular Sums:")
    print([row_sum for row_sum in builder.lower_rows_sum])

    print("\nUpper Triangular")
    for row in builder.upper_rows: print(row)

    print("\nUpper Triangular Sums:")
    print([row_sum for row_sum in builder.upper_rows_sum])
    
    if verbose:

        print("\nDiagonals Upwards")
        for diags in builder.diagonals_up: print(diags)

        print("\nDiagonals Upwards Sums:")
        print([row_sum for row_sum in builder.diagonals_up_sum])

        print("\nDiagonals Downwards")
        for diags in builder.diagonals_down: print(diags)
 
        print("\nDiagonals Downwards Alternating Sums:")
        print([row_sum for row_sum in builder.diagonals_down_altsum])

## <a id='toc1_2_'></a>Riordan

A005043, A126930, A000108, A106640.

In [ ]:
def riordan_generator() -> SeqGenerator:
    b, a, n, r = 1, 0, 1, 0
    yield 1
    while True:
        yield r
        r = n * (2 * a + 3 * b) // (n + 2)
        b, a, n = a, r, n + 1

In [ ]:
Showcase(riordan_generator())

## <a id='toc1_3_'></a>Catalan

A000108, A005043, A007317, A106640.

In [ ]:
def catalan_generator() -> SeqGenerator:
    c, n = 1, 0
    while True:
        yield c
        c = c * (4 * n + 2) // (n + 2)
        n += 1

In [ ]:
Showcase(catalan_generator())

## <a id='toc1_4_'></a>Motzkin

A001006, A126120, A000108, A058987.

In [ ]:
def motzkin_generator() -> SeqGenerator:
    a, b = 1, 1
    yield a
    yield b
    n = 2
    while True:
        m = ((2 * n + 1) * b + (3 * n - 3) * a) // (n + 2)
        yield m
        a, b, n = b, m, n + 1

In [ ]:
Showcase(motzkin_generator())

## <a id='toc1_5_'></a>Fibonacci

A000045, A039834, A001906, A362067, A049601.

In [ ]:
def fibonacci_generator() -> SeqGenerator:
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

In [ ]:
Showcase(fibonacci_generator())

## <a id='toc1_6_'></a>Factorial

A000142, A000166, A000522, A002627, A002467.

In [ ]:
def factorial_generator() -> SeqGenerator:
    a, n = 1, 1
    while True:
        yield a
        a, n = a * n, n + 1

In [ ]:
Showcase(factorial_generator())

## <a id='toc1_7_'></a>Kolakoski

A000002, A054355, A397648.

In [ ]:
def kolakoski_generator() -> SeqGenerator:
    x = y = -1
    while True:
        yield [2, 1][x & 1]
        f = y & ~(y + 1)
        x ^= f
        y = (y + 1) | (f & (x >> 1))

In [ ]:
Showcase(kolakoski_generator())

## <a id='toc1_8_'></a>Bell

A000110, A000296, A005493.

In [ ]:
from itertools import accumulate

def bell_generator() -> SeqGenerator:
    row = [1]
    while True:
        yield row[0]
        row = list(accumulate([row[-1], *row]))

In [ ]:
Showcase(bell_generator())

## <a id='toc1_9_'></a>Fubini

A000670, A052841, A000629, A089677

In [ ]:
def fubini_generator_a() -> SeqGenerator:
    row = [1]
    total, m = 1, 0

    while True:
        yield total
        m += 1  
        row.append(0)  
        total = 0
        for k in range(m, 0, -1):
            val = k * (row[k - 1] + row[k])
            row[k] = val
            total += val
        row[0] = 0

Alternative: this is equivalent, just implemented in C with zip, count and sum. 
/Not/ always faster and requires importing itertools.

In [ ]:
from itertools import count

def fubini_generator() -> SeqGenerator:
    row = [1]
    while True:
        yield sum(row)
        ext = row + [0]   # old row, padded with trailing 0
        row = [0] + [k * (a + b) for k, a, b in zip(count(1), ext, ext[1:])]

In [ ]:
Showcase(fubini_generator())

## <a id='toc1_10_'></a>Central Binomial

A000984, A002426, A026375, A163844, A163774

In [ ]:
def central_binomial_generator() -> SeqGenerator:
    b, n = 1, 0
    while True:
        yield b
        b = b * (4 * n + 2) // (n + 1)
        n += 1

In [ ]:
Showcase(central_binomial_generator())

## <a id='toc1_11_'></a>Subfactorial / Rencontres / Derangements

A000166, A000142, A000023, A002467

In [ ]:
def subfactorial_generator() -> SeqGenerator:
    m, a, n = 1, 1, 0
    while True:
        a, m = a * n + m, -m
        yield a
        n += 1

In [ ]:
Showcase(subfactorial_generator()) 

## <a id='toc1_12_'></a>Involutions (Young tableaux with n cells)

A000085, A005425, A123023, A378100

In [ ]:
def involution_generator() -> SeqGenerator:
    a, b, n = 1, 1, 1
    yield a
    yield b

    while True:
        a, b = b, b + n * a
        n += 1
        yield b

In [ ]:
Showcase(involution_generator()) 

## <a id='toc1_13_'></a>Moebius

A104688, A124839, A008683

In [ ]:
# This is unwise! Moebius(0) is better left undefined. NJAS
#from functools import cache
#@cache
#def mu(n):
#    if n < 2: return n
#    return -sum(mu(d) for d in divisors(n)[:-1])

In [ ]:
from math import isqrt

def moebius_generator() -> SeqGenerator:
    M = [0, 1]
    yield from M

    n = 2
    while True:
        r = isqrt(n)
        s = M[1] 
        
        if n & 1: start, step = 3, 2
        else: start, step = 2, 1

        for d in range(start, r + 1, step):
            q, rem = divmod(n, d)
            if rem == 0:
                s += M[d] + M[q]

        if r * r == n: s -= M[r]

        Mn = -s
        M.append(Mn)
        yield Mn
        n += 1

In [ ]:
Showcase(moebius_generator()) 

## <a id='toc1_14_'></a>Pell

A000129, A016116, A007052, [A077957, A007070]

In mathematics, the Pell numbers are an infinite sequence of integers, known since ancient times, that comprise the denominators of the closest rational approximations to the square root of 2. This sequence of approximations begins ⁠
1/1⁠, ⁠3/2⁠, ⁠7/5⁠, ⁠17/12⁠, and ⁠41/29⁠, so the sequence of Pell numbers begins with 1, 2, 5, 12, and 29. (Wikipedia)

In [ ]:
def pell_generator() -> SeqGenerator:
    a, b = 0, 1
    while True:
        yield b
        a, b = b, a + 2 * b

In [ ]:
Showcase(pell_generator()) 

## <a id='toc1_15_'></a>Jacobsthal

A001045, A000244, A020988

In [ ]:
def jacobsthal_generator() -> SeqGenerator:
    a, b = 0, 1
    while True:
        yield a
        a, b = b, b + 2 * a

In [ ]:
Showcase(jacobsthal_generator()) 

## <a id='toc1_16_'></a>Polya Trees

A000081.

In [ ]:
# This list version is given for comparison with the generator versions below.

def divisor_table(N: int) -> Matrix:
    """divs[j] = list of divisors of j (increasing), for j = 0..N.
    Built with a sieve in O(N log N) -- no factorization needed."""
    divs = [[] for _ in range(N + 1)]
    for d in range(1, N + 1):
        for j in range(d, N + 1, d):
            divs[j].append(d)
    return divs


def A000081_list(N: int) -> Seq:
    divs = divisor_table(N)
    a = [0] * (N + 1)
    b = [0] * (N + 1)     # b[j] = sum_{d|j} d * a[d]

    if N >= 1:
        a[1] = 1
        b[1] = 1 * a[1]

    for n in range(2, N + 1):
        total = 0
        for j in range(1, n):
            total += b[j] * a[n - j]
        a[n] = total // (n - 1)
        b[n] = sum(d * a[d] for d in divs[n])

    return a

In [ ]:
# Assuming a function Divisors(n) is defined elsewhere, which returns the 
# list of divisors of n. Not as efficient as the sieve version given below, 
# but more direct and readable.

def polyatree_gen() -> SeqGenerator:
    a = [0, 1]
    yield a[0]
    yield a[1]

    n = 2
    while True:
        total = 0
        for j in range(1, n):
            inner = 0
            for d in Divisors(j):
                inner += d * a[d]
            total += inner * a[n - j]

        a_n = total // (n - 1) 
        a.append(a_n)
        yield a_n
        n += 1

In [ ]:
def polyatree_generator() -> SeqGenerator:
    """
    Uses the Divisors[]-style formula:
        a[n] = Sum_j ( Sum_{d|j} d*a[d] ) * a[n-j] / (n-1)
    Divisors are built via an incremental sieve instead of trial-division/factoring.
    """
    a = [0, 1]        # a[0], a[1]
    b = [0, 1]        # b[j] = sum_{d|j} d*a[d];  b[1] = 1*a[1]
    divs = [[], [1]]  # divs[j] = list of divisors of j; divs[1] = [1]

    yield a[0]
    yield a[1]

    n = 1
    while True:
        n += 1
        if n >= len(a):   # grow on demand, amortized doubling
            old_len = len(a)
            new_len = max(n + 1, old_len * 2)
            a.extend([0] * (new_len - old_len))
            b.extend([0] * (new_len - old_len))
            divs.extend([[] for _ in range(new_len - old_len)])

            for d in range(1, new_len):
                start = ((old_len + d - 1) // d) * d  # first multiple of d >= old_len
                for j in range(start, new_len, d):
                    divs[j].append(d)

        # --- a[n] via convolution with b[] ---
        total = 0
        for j in range(1, n):
            total += b[j] * a[n - j]
        a[n] = total // (n - 1)
        b[n] = sum(d * a[d] for d in divs[n])

        yield a[n]

In [ ]:
Showcase(polyatree_generator()) 

## <a id='toc1_17_'></a>Euler (alternating permutations, boustrophedon transform)

A000111, A000667, A062162

In [ ]:
def euler_generator() -> SeqGenerator:
    L = [0, 1]      # L[0] = A[-1] = 0, L[1] = A[0] = 1
    offset = 1      # A[k]  <->  L[k + offset]
    k, e, i = 0, 1, 0

    while True:
        Am = 0
        idx = k + e + offset
        if idx == len(L): L.append(0)
        elif idx == -1: L.insert(0, 0); offset += 1
        else: L[idx] = 0
        e = -e

        for _ in range(i + 1):
            pos = k + offset
            Am += L[pos]
            L[pos] = Am
            k += e

        yield Am
        i += 1

In [ ]:
Showcase(euler_generator()) 

## <a id='toc1_18_'></a>Little Schroeder numbers

A001003, A118376, A118376.

In [ ]:
def schroeder_little_generator() -> SeqGenerator:
    b, a, n = 1, 1, 3
    yield b
    yield a

    while True:
        t = a * (6 * n - 9) - (n - 3) * b
        q = t // n
        b, a, n = a, q, n + 1
        yield q

In [ ]:
Showcase(schroeder_little_generator()) 

## <a id='toc1_19_'></a>Big Schroeder numbers

A006318, A174347, A052709

In [ ]:
def schroeder_big_generator() -> SeqGenerator:
    b, a, n = 1, 2, 3
    yield b
    yield a

    while True:
        t = a * (6 * n - 9) - (n - 3) * b
        q = t // n
        b, a, n = a, q, n + 1
        yield q

In [ ]:
Showcase(schroeder_big_generator()) 

## <a id='toc1_20_'></a>Central Delannoy

A001850, A080609, A006139.

In [ ]:
def delannoy_generator() -> SeqGenerator:
    b, a, n = 1, 3, 2
    yield b
    yield a

    while True:
        t = a * (6 * n - 3) - (n - 1) * b
        q = t // n
        b, a, n = a, q, n + 1
        yield q

In [ ]:
Showcase(delannoy_generator()) 

## <a id='toc1_21_'></a>Sets of Lists

A000262, A052844, A052845

In [ ]:
def sets_of_lists_generator() -> SeqGenerator:
    b, a, n = 1, 1, 2
    yield b
    yield a

    while True:
        q = (2 * n - 1) * a - (n - 1) * (n - 2) * b
        b, a, n = a, q, n + 1
        yield q

In [ ]:
Showcase(sets_of_lists_generator()) 

## <a id='toc1_22_'></a>Total Partitions / Ward Set

A000311.

In [ ]:
def total_partitions_generator() -> SeqGenerator:
    yield 0
    yield 1
    row, m = [1], 1

    while True:
        row.append(0)

        for i in range(m, 0, -1):
            row[i] = i * row[i] + (m + i - 1) * row[i - 1]

        row[0] = 0
        m += 1
        yield sum(row)

In [ ]:
Showcase(total_partitions_generator()) 

## <a id='toc1_23_'></a>Ward Cycle

A032188

In [ ]:
def wardcycle_generator() -> SeqGenerator:
    yield 1
    yield 1

    n = 1
    row = [0, 1]

    while True:
        n += 1
        row = row + [0]
        for k in range(n, 0, -1):
            row[k] = (n + k - 1) * (row[k - 1] + row[k])
        yield sum(row)

In [ ]:
Showcase(wardcycle_generator()) 

## Appendix: some matrix generators

In [ ]:
def A060187_mat_generator() -> MatGenerator:
    row = [1]  
    yield list(row)
    n = 1
    while True:
        row.append(1)
        for k in range(n-1, 0, -1):
            row[k] = (2 * (n - k) + 1) * row[k - 1] + (2 * k + 1) * row[k]
        yield list(row)
        n += 1
        
print_matrix(A060187_mat_generator(), n=6)

In [ ]:
def A028338_mat_generator() -> MatGenerator:
    yield list([1])
    n = 1; row = [0, 1]
    while True:
        row.append(1)
        for m in range(n, 0, -1):
            row[m] = (2 * n - 1) * row[m] + row[m - 1]
        yield row[1:]
        n += 1

print_matrix(A028338_mat_generator(), n=6)

# BENCHMARK

In [ ]:
import time

class StopWatch:
    def __init__(
        self,
        comment: str = "elapsed time"
    ) -> None:
        self.start_time = None
        self.text = comment

    def start(self) -> None:
        """Start a new StopWatch"""
        if self.start_time is not None:
            raise RuntimeError("Watch is running. First stop it.")
        self.start_time = time.perf_counter()

    def stop(self) -> float:
        """Stop the StopWatch, and report the elapsed time."""
        if self.start_time is None:
            raise RuntimeError("Watch is not running.")

        elapsed_time = time.perf_counter() - self.start_time
        self.start_time = None

        print(self.text.rjust(17), "{:0.4f}".format(elapsed_time), "sec")

        return elapsed_time


def Benchmark(gen: SeqGenerator, 
              offset:int = 8, 
              size:int = 4
    ) -> list[float]:
    """Benchmark for sequence generators.

    Args:
        gen, sequence generator
        offset > 0, the power of two where the test starts. Defaults to 4.
        size, the length of test run. Defaults to 4.

    Returns:
        List of elapsed time. 
        Stops if the computing time exceeds 1 second.

    Example:
        Benchmark(lambda n, k: n**k)
    """
    print("\n", gen.__qualname__)
    B: list[float] = []
    for s in [2 << n for n in range(offset - 1, offset + size)]:
        t = StopWatch(str(s))
        t.start()
        list(islice(gen, s))
        e = t.stop()
        B.append(e)
        if e > 1.0: break
    return B

In [ ]:
Benchmark(riordan_generator())
Benchmark(catalan_generator())
Benchmark(motzkin_generator())
Benchmark(fibonacci_generator())
Benchmark(factorial_generator())
Benchmark(kolakoski_generator())
Benchmark(bell_generator())
Benchmark(fubini_generator_a())
Benchmark(fubini_generator())
Benchmark(central_binomial_generator())
Benchmark(subfactorial_generator())
Benchmark(involution_generator())
Benchmark(moebius_generator())
Benchmark(pell_generator())
Benchmark(jacobsthal_generator())
Benchmark(polyatree_generator())
Benchmark(euler_generator())
Benchmark(schroeder_little_generator())
Benchmark(schroeder_big_generator())
Benchmark(delannoy_generator())
Benchmark(sets_of_lists_generator())
Benchmark(total_partitions_generator())
Benchmark(wardcycle_generator())